# Fatigue modeling

Train ordinal and classification models on processed physical-activity data with participant-level splits and GroupKFold CV.

In [ ]:
%pip install -q -r ../../requirements.txt

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import lightgbm as lgb
import mord

DATA_PATH = "../../mcphases/merged/physical_activity_merged_processed.csv"
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_CV_FOLDS = 5
HIGH_FATIGUE_THRESHOLD = 4
PHASE_ORDER = ["Menstrual", "Follicular", "Fertility", "Luteal"]


## 1. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(df['fatigue_num'].value_counts().sort_index().to_frame('count'))


## 2. Features and targets

In [ ]:
NUMERIC_FEATURES = [
    'lightly', 'moderately', 'very', 'calories_sum',
    'filtered_demographic_vo2_max',
    'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone',
    'lh_smooth', 'estrogen_smooth',
    'age_of_first_menarche', 'age',
    'menstrual_health_literacy_num', 'sexually_active_num',
]
CATEGORICAL_FEATURES = ['is_weekend', 'phase', 'exerciselevel_num']
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

y_ordinal = df['fatigue_num'].astype(int)
y_high_fatigue = (df['fatigue_num'] >= HIGH_FATIGUE_THRESHOLD).astype(int)
groups = df['id']

print(f"High-fatigue rate (>= {HIGH_FATIGUE_THRESHOLD}): {y_high_fatigue.mean():.3f}")


## 3. Participant-level train/val vs test split

In [ ]:
def split_participant_ids(unique_ids, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    ids = np.array(sorted(unique_ids))
    rng.shuffle(ids)
    n_test = max(1, int(round(len(ids) * test_size)))
    test_ids = set(ids[:n_test])
    train_val_ids = set(ids[n_test:])
    return train_val_ids, test_ids


train_val_ids, test_ids = split_participant_ids(df['id'].unique(), TEST_SIZE, RANDOM_STATE)
assert train_val_ids.isdisjoint(test_ids)

train_val_mask = df['id'].isin(train_val_ids)
test_mask = df['id'].isin(test_ids)

split_summary = pd.DataFrame([
    {
        'split': 'train_val',
        'participants': len(train_val_ids),
        'rows': int(train_val_mask.sum()),
        'high_fatigue_rate': y_high_fatigue[train_val_mask].mean(),
    },
    {
        'split': 'test',
        'participants': len(test_ids),
        'rows': int(test_mask.sum()),
        'high_fatigue_rate': y_high_fatigue[test_mask].mean(),
    },
])
display(split_summary)
print('Test participant ids:', sorted(test_ids))


## 4. Preprocessing helpers

In [ ]:
def make_feature_matrix(data, numeric_features=NUMERIC_FEATURES, categorical_features=CATEGORICAL_FEATURES):
    X = data[numeric_features + categorical_features].copy()
    if 'phase' in X.columns:
        X['phase'] = pd.Categorical(X['phase'], categories=PHASE_ORDER, ordered=True)
    if 'is_weekend' in X.columns:
        X['is_weekend'] = X['is_weekend'].astype(int)
    return X


def build_tree_matrix(X):
    X_tree = X.copy()
    if 'phase' in X_tree.columns:
        X_tree['phase'] = X_tree['phase'].astype(str)
    return pd.get_dummies(X_tree, columns=['phase'], prefix='phase', dtype=int)


X_all = make_feature_matrix(df)
X_train_val = X_all.loc[train_val_mask].reset_index(drop=True)
X_test = X_all.loc[test_mask].reset_index(drop=True)

y_ord_train_val = y_ordinal.loc[train_val_mask].reset_index(drop=True)
y_ord_test = y_ordinal.loc[test_mask].reset_index(drop=True)
y_clf_train_val = y_high_fatigue.loc[train_val_mask].reset_index(drop=True)
y_clf_test = y_high_fatigue.loc[test_mask].reset_index(drop=True)
groups_train_val = groups.loc[train_val_mask].reset_index(drop=True)

X_train_val_tree = build_tree_matrix(X_train_val)
X_test_tree = build_tree_matrix(X_test)


## 5. Metrics and CV framework

In [ ]:
def clip_ordinal_predictions(preds, low=0, high=5):
    return np.clip(np.rint(preds), low, high).astype(int)


def compute_metrics(y_true, y_pred, task='ordinal'):
    y_true = np.asarray(y_true)
    if task == 'ordinal':
        y_pred = clip_ordinal_predictions(y_pred)
        rmse = mean_squared_error(y_true, y_pred) ** 0.5
        return {
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': rmse,
            'r2': r2_score(y_true, y_pred),
            'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        }
    if task == 'classification':
        y_pred = np.asarray(y_pred).astype(int)
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            'precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            'recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            'positive_rate': float(y_true.mean()),
        }
    raise ValueError(f"Unknown task: {task}")


def summarize_fold_metrics(fold_df, metric_cols):
    summary = {'mean': fold_df[metric_cols].mean(), 'std': fold_df[metric_cols].std()}
    return pd.DataFrame(summary).T


def run_group_cv(model_factory, X, y, groups, n_splits=5, task='ordinal'):
    gkf = GroupKFold(n_splits=n_splits)
    fold_results = []
    test_id_set = set(test_ids)

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        train_group_ids = set(groups.iloc[train_idx])
        val_group_ids = set(groups.iloc[val_idx])
        assert train_group_ids.isdisjoint(val_group_ids)
        assert train_group_ids.isdisjoint(test_id_set)
        assert val_group_ids.isdisjoint(test_id_set)

        model = model_factory()
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[val_idx])
        metrics = compute_metrics(y.iloc[val_idx], preds, task=task)
        metrics['fold'] = fold
        metrics['n_train_participants'] = len(train_group_ids)
        metrics['n_val_participants'] = len(val_group_ids)
        if task == 'classification':
            metrics['val_positive_rate'] = float(y.iloc[val_idx].mean())
        fold_results.append(metrics)

    fold_df = pd.DataFrame(fold_results)
    metric_cols = [c for c in fold_df.columns if c not in {'fold', 'n_train_participants', 'n_val_participants', 'val_positive_rate'}]
    return fold_df, summarize_fold_metrics(fold_df, metric_cols)


def evaluate_on_test(model_factory, X_train_val, y_train_val, X_test, y_test, task='ordinal'):
    model = model_factory()
    model.fit(X_train_val, y_train_val)
    preds = model.predict(X_test)
    return compute_metrics(y_test, preds, task=task)


def run_model_benchmark(name, model_factory, X_train_val, y_train_val, groups, X_test, y_test, task='ordinal', n_splits=5):
    try:
        fold_df, cv_summary = run_group_cv(model_factory, X_train_val, y_train_val, groups, n_splits=n_splits, task=task)
        test_metrics = evaluate_on_test(model_factory, X_train_val, y_train_val, X_test, y_test, task=task)
        return {
            'status': 'ok',
            'name': name,
            'fold_df': fold_df,
            'cv_summary': cv_summary,
            'test_metrics': test_metrics,
        }
    except NotImplementedError as exc:
        return {'status': 'stub', 'name': name, 'message': str(exc)}


## 6. Model builders

In [ ]:
class StubModel:
    def __init__(self, message):
        self.message = message

    def fit(self, X, y):
        raise NotImplementedError(self.message)

    def predict(self, X):
        raise NotImplementedError(self.message)


def build_ordered_logistic():
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ('prep', preprocessor),
        ('model', mord.LogisticAT(alpha=1.0)),
    ])


def build_ordinal_rf():
    return StubModel('Ordinal RF stub: RandomForestRegressor + round/clip, or dedicated ordinal library.')


def build_catboost_ordinal():
    return StubModel('CatBoost ordinal stub: CatBoostRegressor with RMSE + round/clip, or ordinal loss.')


def build_mixed_effects():
    return StubModel('Mixed effects stub: statsmodels OrderedModel with random intercept by participant id.')


def build_lstm():
    return StubModel('LSTM/GRU stub: per-id sequences by day_in_study; requires torch + sequence CV runner.')


def build_lightgbm_classifier():
    return lgb.LGBMClassifier(
        objective='binary',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_estimators=200,
        verbose=-1,
    )


def build_rf_classifier():
    return RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_estimators=300,
    )


ORDINAL_MODELS = {
    'ordered_logistic': build_ordered_logistic,
    'ordinal_rf': build_ordinal_rf,
    'catboost_ordinal': build_catboost_ordinal,
    'mixed_effects': build_mixed_effects,
    'lstm': build_lstm,
}

CLASSIFICATION_MODELS = {
    'lightgbm': build_lightgbm_classifier,
    'random_forest': build_rf_classifier,
}


## 7. Ordinal track

In [ ]:
ordinal_results = []
for name, factory in ORDINAL_MODELS.items():
    result = run_model_benchmark(
        name,
        factory,
        X_train_val,
        y_ord_train_val,
        groups_train_val,
        X_test,
        y_ord_test,
        task='ordinal',
        n_splits=N_CV_FOLDS,
    )
    ordinal_results.append(result)
    if result['status'] == 'stub':
        print(f"[stub] {name}: {result['message']}")
    else:
        print(f"[ok] {name}")
        display(result['cv_summary'])
        display(pd.DataFrame([result['test_metrics']], index=[f'{name}_test']))


## 8. Classification track (high fatigue >= 4)

In [ ]:
classification_results = []
for name, factory in CLASSIFICATION_MODELS.items():
    result = run_model_benchmark(
        name,
        factory,
        X_train_val_tree,
        y_clf_train_val,
        groups_train_val,
        X_test_tree,
        y_clf_test,
        task='classification',
        n_splits=N_CV_FOLDS,
    )
    classification_results.append(result)
    if result['status'] == 'stub':
        print(f"[stub] {name}: {result['message']}")
    else:
        print(f"[ok] {name}")
        display(result['cv_summary'])
        display(pd.DataFrame([result['test_metrics']], index=[f'{name}_test']))


## 9. Results summary

In [ ]:
def collect_summaries(results, task='ordinal'):
    cv_rows, test_rows = [], []
    metric_cols = ['mae', 'rmse', 'r2', 'qwk'] if task == 'ordinal' else ['accuracy', 'f1', 'precision', 'recall']
    for result in results:
        if result['status'] != 'ok':
            continue
        cv_mean = result['cv_summary'].loc['mean', metric_cols]
        cv_std = result['cv_summary'].loc['std', metric_cols]
        cv_row = {'model': result['name']}
        for col in metric_cols:
            cv_row[f'cv_{col}'] = cv_mean[col]
            cv_row[f'cv_{col}_std'] = cv_std[col]
        cv_rows.append(cv_row)

        test_row = {'model': result['name']}
        for col in metric_cols:
            test_row[f'test_{col}'] = result['test_metrics'][col]
        test_rows.append(test_row)

    return pd.DataFrame(cv_rows).set_index('model'), pd.DataFrame(test_rows).set_index('model')


ordinal_cv_summary, ordinal_test_summary = collect_summaries(ordinal_results, task='ordinal')
clf_cv_summary, clf_test_summary = collect_summaries(classification_results, task='classification')

print('Ordinal CV summary')
display(ordinal_cv_summary)
print('Ordinal held-out test summary')
display(ordinal_test_summary)
print('Classification CV summary')
display(clf_cv_summary)
print('Classification held-out test summary')
display(clf_test_summary)
